In [1]:
%%writefile servidor.c
#include <stdio.h>
#include <netdb.h>
#include <netinet/in.h>
#include <stdlib.h>
#include <string.h>
#include <sys/socket.h>
#include <sys/types.h>
#include <unistd.h> // read(), write(), close()

#define MAX 80
#define PORT 8080
#define SA struct sockaddr

void func(int connfd) {
    char buff[MAX];
    int n;
    for (;;) {
        bzero(buff, MAX);
        read(connfd, buff, sizeof(buff));
        printf("Del cliente: %s\t: ", buff);
        bzero(buff, MAX);
        n = 0;
        while ((buff[n++] = getchar()) != '\n');
        write(connfd, buff, sizeof(buff));
        if (strncmp("SALIR", buff, 4) == 0) {
            printf("Salgo del servidor...\n");
            break;
        }
    }
}

int main() {
    int sockfd, connfd, len;
    struct sockaddr_in servaddr, cli;
    
    sockfd = socket(AF_INET, SOCK_STREAM, 0);
    if (sockfd == -1) {
        printf("Falla la creación del socket...\n");
        exit(0);
    }
    printf("Socket creado...\n");
    bzero(&servaddr, sizeof(servaddr));
    
    servaddr.sin_family = AF_INET;
    servaddr.sin_addr.s_addr = htonl(INADDR_ANY);
    servaddr.sin_port = htons(PORT);
    
    if ((bind(sockfd, (SA*)&servaddr, sizeof(servaddr))) != 0) {
        printf("Falla socket bind ...\n");
        exit(0);
    }
    printf("Se hace el socket bind ..\n");
    
    if ((listen(sockfd, 5)) != 0) {
        printf("Falla el listen ...\n");
        exit(0);
    }
    printf("Servidor en modo escucha ...\n");
    len = sizeof(cli);
    
    connfd = accept(sockfd, (SA*)&cli, &len);
    if (connfd < 0) {
        printf("Falla al aceptar datos en el servidor...\n");
        exit(0);
    }
    printf("El servidor acepta al cliente ...\n");
    
    func(connfd);
    close(sockfd);
}

Overwriting servidor.c


In [2]:
%%writefile servidor_concurrente.c
#include <stdio.h>
#include <netdb.h>
#include <netinet/in.h>
#include <stdlib.h>
#include <string.h>
#include <sys/socket.h>
#include <sys/types.h>
#include <unistd.h> // read(), write(), close()
#include <arpa/inet.h>
#include <signal.h>

#define MAX 1024
#define PORT 8080

void handle_client(int client_fd, struct sockaddr_in client_addr) {
    char buffer[MAX];
    char name[50] = "Cliente";
    char client_ip[INET_ADDRSTRLEN];
    char response[MAX + 100];
    pid_t pid = getpid();

    inet_ntop(AF_INET, &(client_addr.sin_addr), client_ip, INET_ADDRSTRLEN);
    int client_port = ntohs(client_addr.sin_port);

    printf("[PID %d] Conexión desde %s:%d\n", pid, client_ip, client_port);

    // Recibir nombre del cliente
    recv(client_fd, name, sizeof(name), 0);
    printf("[PID %d] Cliente: %s\n", pid, name);

    while (1) {
        memset(buffer, 0, MAX);
        int bytes = recv(client_fd, buffer, MAX, 0);
        if (bytes <= 0) {
            printf("[PID %d] %s se desconectó.\n", pid, name);
            break;
        }

        // Verificar si cliente quiere finalizar
        if (strncmp(buffer, "FIN", 3) == 0) {
            printf("[PID %d] %s finalizó la conexión.\n", pid, name);
            break;
        }

        printf("[%s | PID %d]: %s", name, pid, buffer);

        // Preparar respuesta automática
        snprintf(response, sizeof(response),
                 "Este es el mensaje recibido por el servidor: %s", buffer);
        send(client_fd, response, strlen(response), 0);
    }

    close(client_fd);
    printf("[PID %d] Proceso hijo termina.\n", pid);
    exit(0);
}

int main() {
    int server_fd, client_fd;
    struct sockaddr_in server_addr, client_addr;

    signal(SIGCHLD, SIG_IGN); // Evita procesos zombies

    server_fd = socket(AF_INET, SOCK_STREAM, 0);
    if (server_fd == -1) {
        perror("Error al crear socket");
        exit(1);
    }

    server_addr.sin_family = AF_INET;
    server_addr.sin_addr.s_addr = INADDR_ANY;
    server_addr.sin_port = htons(PORT);
    memset(&(server_addr.sin_zero), 0, 8);

    if (bind(server_fd, (struct sockaddr*)&server_addr, sizeof(server_addr)) != 0) {
        perror("Error en bind");
        exit(1);
    }

    if (listen(server_fd, 5) != 0) {
        perror("Error en listen");
        exit(1);
    }

    printf("Servidor concurrente activo en puerto %d...\n", PORT);

    socklen_t len = sizeof(client_addr);
    while (1) {
        client_fd = accept(server_fd, (struct sockaddr*)&client_addr, &len);
        if (client_fd < 0) {
            perror("Fallo en accept");
            continue;
        }

        pid_t pid = fork();
        if (pid == 0) {
            close(server_fd);
            handle_client(client_fd, client_addr);
        } else if (pid > 0) {
            printf("[PID %d] Cliente conectado. Proceso hijo PID %d\n", getpid(), pid);
            close(client_fd);
        } else {
            perror("Error en fork");
            close(client_fd);
        }
    }

    close(server_fd);
    return 0;
}

Overwriting servidor_concurrente.c


In [3]:
%%writefile servidor_con_select.c
#include <stdio.h>
#include <netdb.h>
#include <netinet/in.h>
#include <stdlib.h>
#include <string.h>
#include <sys/socket.h>
#include <sys/types.h>
#include <unistd.h> // read(), write(), close()
#include <arpa/inet.h>
#include <signal.h>
#include <errno.h>

#define PORT 12345
#define MAX_CLIENTS 10
#define BUFFER_SIZE 1024

int main() {
    int server_fd, new_socket, client_sockets[MAX_CLIENTS];
    struct sockaddr_in address;
    fd_set read_fds;
    int max_sd, sd, activity, valread;
    char buffer[BUFFER_SIZE];
    
    // Inicializar todos los clientes a 0
    for (int i = 0; i < MAX_CLIENTS; i++) {
        client_sockets[i] = 0;
    }

    // Crear socket
    if ((server_fd = socket(AF_INET, SOCK_STREAM, 0)) == 0) {
        perror("socket falló");
        exit(EXIT_FAILURE);
    }

    // Configurar dirección del servidor
    address.sin_family = AF_INET;
    address.sin_addr.s_addr = INADDR_ANY;
    address.sin_port = htons(PORT);

    // Enlazar socket al puerto
    if (bind(server_fd, (struct sockaddr *)&address, sizeof(address)) < 0) {
        perror("bind falló");
        exit(EXIT_FAILURE);
    }

    // Escuchar conexiones entrantes
    if (listen(server_fd, 3) < 0) {
        perror("listen");
        exit(EXIT_FAILURE);
    }

    printf("Servidor escuchando en puerto %d...\n", PORT);

    while (1) {
        // Limpiar conjunto de descriptores
        FD_ZERO(&read_fds);
        FD_SET(server_fd, &read_fds);
        max_sd = server_fd;

        // Agregar sockets de clientes al conjunto
        for (int i = 0; i < MAX_CLIENTS; i++) {
            sd = client_sockets[i];
            if (sd > 0) FD_SET(sd, &read_fds);
            if (sd > max_sd) max_sd = sd;
        }

        // Esperar actividad
        activity = select(max_sd + 1, &read_fds, NULL, NULL, NULL);

        if ((activity < 0) && (errno != EINTR)) {
            printf("Error en select\n");
        }

        // Nueva conexión
        if (FD_ISSET(server_fd, &read_fds)) {
            socklen_t addrlen = sizeof(address);
            if ((new_socket = accept(server_fd, (struct sockaddr *)&address, &addrlen)) < 0) {
                perror("accept");
                exit(EXIT_FAILURE);
            }

            printf("Nueva conexión: socket fd %d, IP %s, puerto %d\n",
                   new_socket, inet_ntoa(address.sin_addr), ntohs(address.sin_port));

            // Agregar a la lista de clientes
            for (int i = 0; i < MAX_CLIENTS; i++) {
                if (client_sockets[i] == 0) {
                    client_sockets[i] = new_socket;
                    break;
                }
            }
        }

        // Revisar mensajes de clientes
        for (int i = 0; i < MAX_CLIENTS; i++) {
            sd = client_sockets[i];

            if (FD_ISSET(sd, &read_fds)) {
                if ((valread = read(sd, buffer, BUFFER_SIZE)) == 0) {
                    // Desconexión
                    socklen_t addrlen = sizeof(address);
                    getpeername(sd, (struct sockaddr*)&address, (socklen_t*)&addrlen);
                    printf("Cliente desconectado: IP %s, puerto %d\n",
                           inet_ntoa(address.sin_addr), ntohs(address.sin_port));
                    close(sd);
                    client_sockets[i] = 0;
                } else {
                    buffer[valread] = '\0';
                    printf("Mensaje recibido: %s", buffer);

                    // Reenviar mensaje a otros clientes
                    for (int j = 0; j < MAX_CLIENTS; j++) {
                        if (client_sockets[j] != 0 && client_sockets[j] != sd) {
                            send(client_sockets[j], buffer, strlen(buffer), 0);
                        }
                    }
                }
            }
        }
    }

    return 0;
}

Overwriting servidor_con_select.c


In [4]:
%%writefile cliente.c
#include <arpa/inet.h> // inet_addr()
#include <netdb.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <strings.h> // bzero()
#include <sys/socket.h>
#include <unistd.h> // read(), write(), close()
#define MAX 80
#define PORT 8080
#define SA struct sockaddr

void func(int sockfd) {
    char buff[MAX];
    int n;
    for (;;) {
        bzero(buff, sizeof(buff));
        printf("Ingrese texto : ");
        n = 0;
        while ((buff[n++] = getchar()) != '\n');
        write(sockfd, buff, sizeof(buff));
        bzero(buff, sizeof(buff));
        read(sockfd, buff, sizeof(buff));
        printf("Servidor : %s", buff);
        if ((strncmp(buff, "exit", 4)) == 0) {
            printf("Cliente cierra conexión...\n");
            break;
        }
    }
}

int main() {
    int sockfd;
    struct sockaddr_in servaddr;

    sockfd = socket(AF_INET, SOCK_STREAM, 0);
    if (sockfd == -1) {
        printf("Falla la creación del socket...\n");
        exit(0);
    }
    else
        printf("Socket creado ..\n");
    bzero(&servaddr, sizeof(servaddr));

    servaddr.sin_family = AF_INET;
    servaddr.sin_addr.s_addr = inet_addr("127.0.0.1");
    servaddr.sin_port = htons(PORT);

    if (connect(sockfd, (SA*)&servaddr, sizeof(servaddr)) != 0) {
        printf("Falla de conexión con servidor...\n");
        exit(0);
    }
    else
        printf("Conectado al servidor..\n");

    func(sockfd);
    close(sockfd);
}

Overwriting cliente.c


In [5]:
%%writefile cliente_nobloqueante.c
#include <arpa/inet.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <sys/socket.h>
#include <unistd.h>
#include <fcntl.h> // Necesario para fcntl()
#include <errno.h> // Necesario para manejar EWOULDBLOCK

#define PORT 12345
#define BUFFER_SIZE 1024

int main() {
    int sock;
    struct sockaddr_in server_addr;
    char buffer[BUFFER_SIZE];

    sock = socket(AF_INET, SOCK_STREAM, 0);
    if (sock == -1) {
        perror("Error al abrir el socket");
        exit(EXIT_FAILURE);
    }

    server_addr.sin_family = AF_INET;
    server_addr.sin_port = htons(PORT);
    server_addr.sin_addr.s_addr = inet_addr("127.0.0.1");

    if (connect(sock, (struct sockaddr *)&server_addr, sizeof(server_addr)) < 0) {
        perror("Error al conectar");
        exit(EXIT_FAILURE);
    }

    int flags = fcntl(sock, F_GETFL, 0);
    fcntl(sock, F_SETFL, flags | O_NONBLOCK);
    
    int stdin_flags = fcntl(STDIN_FILENO, F_GETFL, 0);
    fcntl(STDIN_FILENO, F_SETFL, stdin_flags | O_NONBLOCK);

    printf("Modo no bloqueante activo. Escribe y espera respuestas...\n");

    while (1) {
        if (fgets(buffer, BUFFER_SIZE, stdin) != NULL) {
            send(sock, buffer, strlen(buffer), 0);
            printf("> ");
            fflush(stdout);
        }

        memset(buffer, 0, BUFFER_SIZE);
        int valread = recv(sock, buffer, BUFFER_SIZE - 1, 0);

        if (valread > 0) {
            printf("\nServidor: %s\n> ", buffer);
            fflush(stdout);
        } else if (valread == 0) {
            printf("\nConexión cerrada por el servidor.\n");
            break;
        } else {
            if (errno != EWOULDBLOCK && errno != EAGAIN) {
                perror("Error en lectura");
                break;
            }
        }

        usleep(100000); // 100ms
    }

    close(sock);
    return 0;
}

Overwriting cliente_nobloqueante.c


In [6]:
%%writefile cliente_con_select.c
#include <arpa/inet.h> // inet_addr()
#include <netdb.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <strings.h> // bzero()
#include <sys/socket.h>
#include <unistd.h> // read(), write(), close()
#define PORT 12345
#define BUFFER_SIZE 1024

int main() {
    int sock;
    struct sockaddr_in server_addr;
    char buffer[BUFFER_SIZE];
    fd_set read_fds;

    // Crear socket
    sock = socket(AF_INET, SOCK_STREAM, 0);
    if (sock < 0) {
        perror("socket");
        exit(EXIT_FAILURE);
    }

    // Configurar dirección del servidor
    server_addr.sin_family = AF_INET;
    server_addr.sin_port = htons(PORT);
    inet_pton(AF_INET, "127.0.0.1", &server_addr.sin_addr);

    // Conectarse al servidor
    if (connect(sock, (struct sockaddr *)&server_addr, sizeof(server_addr)) < 0) {
        perror("connect");
        exit(EXIT_FAILURE);
    }

    printf("Conectado al servidor. Puedes escribir mensajes:\n");

    while (1) {
        FD_ZERO(&read_fds);
        FD_SET(0, &read_fds);      // Entrada estándar
        FD_SET(sock, &read_fds);   // Socket

        if (select(sock + 1, &read_fds, NULL, NULL, NULL) < 0) {
            perror("select");
            exit(EXIT_FAILURE);
        }

        // Entrada del usuario
        if (FD_ISSET(0, &read_fds)) {
            fgets(buffer, BUFFER_SIZE, stdin);
            send(sock, buffer, strlen(buffer), 0);
        }

        // Mensaje del servidor
        if (FD_ISSET(sock, &read_fds)) {
            int valread = read(sock, buffer, BUFFER_SIZE - 1);
            if (valread <= 0) {
                printf("Desconectado del servidor.\n");
                break;
            }
            buffer[valread] = '\0';
            printf("Mensaje: %s", buffer);
        }
    }

    close(sock);
    return 0;
}

Overwriting cliente_con_select.c


In [7]:
%%writefile cliente_bloqueante.c
#include <arpa/inet.h> // inet_addr()
#include <netdb.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <strings.h> // bzero()
#include <sys/socket.h>
#include <unistd.h> // read(), write(), close()
#define PORT 12345
#define BUFFER_SIZE 1024

int main() {
    int sock;
    struct sockaddr_in server_addr;
    char buffer[BUFFER_SIZE];

    // 1. Crear socket
    sock = socket(AF_INET, SOCK_STREAM, 0);
    if (sock == -1) {
        perror("Error al abrir el socket ...");
        exit(EXIT_FAILURE);
    }

    // 2. Configurar dirección del servidor
    server_addr.sin_family = AF_INET;
    server_addr.sin_port = htons(PORT);
    server_addr.sin_addr.s_addr = inet_addr("127.0.0.1");

    // 3. Conectar al servidor
    if (connect(sock, (struct sockaddr *)&server_addr, sizeof(server_addr)) < 0) {
        perror("Error al conectar");
        exit(EXIT_FAILURE);
    }

    printf("Conectado al servidor. Escribe mensajes y presiona Enter.\n");

    // 4. Bucle principal bloqueante
    while (1) {
        printf("> ");
        fgets(buffer, BUFFER_SIZE, stdin);  // BLOQUEA esperando entrada
        send(sock, buffer, strlen(buffer), 0);

        // Leer respuesta del servidor
        int valread = read(sock, buffer, BUFFER_SIZE - 1);
        if (valread <= 0) {
            printf("Conexión cerrada por el servidor.\n");
            break;
        }

        buffer[valread] = '\0';
        printf("Servidor: %s", buffer);
    }

    close(sock);
    return 0;
}

Overwriting cliente_bloqueante.c


In [8]:
!gcc servidor.c -o servidor

servidor.c:61:40: warning: passing 'int *' to parameter of type 'socklen_t *' (aka 'unsigned int *') converts between pointers to integer types with different sign [-Wpointer-sign]
   61 |     connfd = accept(sockfd, (SA*)&cli, &len);
      |                                        ^~~~
/Library/Developer/CommandLineTools/SDKs/MacOSX.sdk/usr/include/sys/socket.h:708:73: note: passing argument to parameter here
  708 | int     accept(int, struct sockaddr * __restrict, socklen_t * __restrict)
      |                                                                         ^
1 warning generated.


In [9]:
!gcc servidor_concurrente.c -o servidor_concurrente

In [10]:
!gcc servidor_con_select.c -o servidor_con_select

In [11]:
!gcc cliente.c -o cliente

In [12]:
!gcc cliente_no_bloqueante.c -o cliente_no_bloqueante

In [13]:
!gcc cliente_con_select.c -o cliente_con_select

In [14]:
!gcc cliente_bloqueante.c -o cliente_bloqueante